In [11]:
from sklearn.metrics import mutual_info_score
from os import listdir
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy, pearsonr
from dit import Distribution
from dit.multivariate import coinformation
import seaborn as sns
from collections import Counter
import scipy.signal as signal 
new_rc_params = {'text.usetex': False,
"svg.fonttype": 'none'
}
plt.rcParams.update(new_rc_params)

In [2]:
def load_data_torque(directory):
    torques = []
    stim_id = []
    for f in listdir(directory):
        torques.append(np.genfromtxt(directory + '/' + f, delimiter=','))
        stim_id.append(Path(f).stem)
    torques = np.asarray(torques)
    return torques, stim_id

In [3]:
def load_data(directory, population, var_of_interest):
    neural_act = []
    stim_id = []
    for f in listdir('./neural_activity/Scale1/' + directory + population + var_of_interest):
        neural_act.append(np.genfromtxt('./neural_activity/Scale1/' + directory + population + var_of_interest + '/' + f, delimiter=','))
        stim_id.append(Path(f).stem)
    neural_act = np.asarray(neural_act)
    return neural_act, np.asarray(stim_id, dtype=int)

In [4]:
def load_spiketimes(directory, population, rank):
    neural_act = []
    stim_id = []
    for f in listdir('./neural_activity/Scale1/' + directory + population + '/spike_time'):
        sp_time = np.genfromtxt('./neural_activity/Scale1/' + directory + population + '/spike_time' + '/' + f, delimiter=',')
        sp_time = np.unique(sp_time)[rank]
        neural_act.append(sp_time)
        stim_id.append(Path(f).stem)
    neural_act = np.asarray(neural_act)
    return neural_act, np.asarray(stim_id, dtype=int)


In [5]:
def load_spiketime_per_neuron(directory, population, populationsize, rank):
    neuron_ids, stim_id = load_data(directory, population, '/spike_id')
    neuron_sptime, _ = load_data(directory, population, '/spike_time')
    sp_time_per_neuron = np.zeros((len(stim_id), populationsize))
    for j in range(len(stim_id)):
        #sp_time_per_neuron_per_stim = []
        for i in range(populationsize):
            #sp_time_per_neuron_per_stim.append(neuron_sptime[j][np.where(neuron_ids[j] == i)])
            try:
                sp_time_per_neuron[j, i] = neuron_sptime[j][np.where(neuron_ids[j] == i)][rank]
            except:
                pass
        #sp_time_per_neuron.append(sp_time_per_neuron_per_stim)
    return sp_time_per_neuron, stim_id

In [6]:
def compute_nb_spike_per_neuron(data, popsize):
    data = np.asarray(data, dtype=int)
    spike_per_neuron = np.zeros(popsize)
    for i in np.unique(data):
        spike_per_neuron[i] = np.sum(np.where(data == i, 1, 0))

    return spike_per_neuron

In [7]:
def compute_amplitude(torques):
    amplitude = []
    for i in range(len(torques)):
        amplitude.append(np.sqrt(torques[i][1]**2 + torques[i][2]**2))
    return np.asarray(amplitude)

def compute_amplitude_ONOFF(amplitude, th):
    amplitude_ONOFF = np.zeros(len(amplitude))
    th_reached = np.argwhere(amplitude > th)
    amplitude_ONOFF[th_reached[0][0]:th_reached[-1][0]] += 1
    return amplitude_ONOFF

def compute_direction(direction, amplitude):
    direction_array = np.ones(len(amplitude)) * direction
    return direction_array

In [ ]:
torque, stim_id = load_data_torque('./0_frequency/Moments')

In [20]:
plt.plot(np.abs(signal.envelope(compute_amplitude(torque[0, 1:]))))
plt.plot(compute_amplitude(torque[0, 1:]))

AttributeError: module 'scipy.signal' has no attribute 'envelope'